In [13]:
%load_ext IPython.extensions.autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [14]:
import sys
sys.path.append('../../')
from model import FinData
from model import train_valid_test_split
from model import CatboostFinModel, SVMFinModel
from model import precision_long, recall_long, fbeta_metric_long, precision_short, recall_short, fbeta_metric_short

import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import pandas as pd
import numpy as np

import json
import datetime as dt
import pandas as pd
import optuna

In [15]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

def compute_precision_recall(group):
    y_true = group['target']
    y_pred = group['pred']
    close = group['close']
    accuracy = accuracy_score(y_true, y_pred)
    precision_1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    precision_0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
    recall_1 = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    recall_0 = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    fbeta_long = fbeta_metric_long(y_pred, close, commission=0.0008, beta=0.0005)
    fbeta_short = fbeta_metric_short( y_pred, close, commission=0.0008, beta=0.0005)
    pr_long = precision_long(y_pred, close, commission=0.000)
    pr_short = precision_short(y_pred, close, commission=0.0008)
    re_long = recall_long(y_pred, close, commission=0.000)
    re_short = recall_short(y_pred, close, commission=0.0008)
    return pd.Series({
        'precision_1': precision_1,
        'precision_0': precision_0,
        'recall_1': recall_1,
        'recall_0': recall_0,
        'accuracy':  accuracy,
        'fbeta_long': fbeta_long,
        'fbeta_short': fbeta_short,
        'pr_long': pr_long,
        'pr_short': pr_short,
        're_long': re_long,
        're_short': re_short,
    })



In [23]:
import pandas as pd
import glob
from sklearn.metrics import accuracy_score, precision_score, recall_score, fbeta_score

all_summary_dict = dict()
commission_rate = 0.0008
b = 0.005



for model_type in ['catboost', 'lightgbm', 'xgboost']:
    csv_paths = glob.glob(f'../../generated_datasets/{model_type}_no_weights/*.csv')  # или свой путь


    for path in csv_paths:
        data = pd.read_csv(path)

        data['utc'] = pd.to_datetime(data['utc'])
        data = data.sort_values('utc')

        threshold = 0.5
        data['pred'] = (data['predicted_proba'] > threshold).astype(int)
        data['group_id'] = ((data['utc'] - data['utc'].min()).dt.days // 5)
        data['target'] = (data['close'].shift(-1) >= data['close']).astype(int)

        name = path.split('/')[-1].split('_')[0]
        if name not in all_summary_dict:
            all_summary_dict[name] = []

        metrics_by_group = data.groupby('group_id').apply(compute_precision_recall).reset_index()

        type = path.split('/')[-1].split('_')[1].split('.')[0]
        type_bool = 1 if type == 'long' else 0

        fname = path.split('/')[-1].split('_')[0] + '_' + type + '_' + model_type + '_no_weight'


        row = {
            'file': fname,
            'accuracy_mean': metrics_by_group['accuracy'].mean(),
            'accuracy_median': metrics_by_group['accuracy'].median(),
            'pr_mean': metrics_by_group[f'precision_{type_bool}'].mean(),
            'pr_median': metrics_by_group[f'precision_{type_bool}'].median(),
            're_mean': metrics_by_group[f'recall_{type_bool}'].mean(),
            're_median': metrics_by_group[f'recall_{type_bool}'].median(),
            'pr_custom_mean': metrics_by_group[f'pr_{type}'].mean(),
            're_custom_mean': metrics_by_group[f're_{type}'].mean(),
            'f_beta_custom_mean': metrics_by_group[f'fbeta_{type}'].mean(),
            'pr_custom_median': metrics_by_group[f'pr_{type}'].median(),
            're_custom_median': metrics_by_group[f're_{type}'].median(),
            'f_beta_custom_median': metrics_by_group[f'fbeta_{type}'].median(),
        }
        all_summary_dict[name].append(row)

for model_type in ['catboost', 'lightgbm', 'xgboost']:
    csv_paths = glob.glob(f'../../generated_datasets/{model_type}_with_weights/*.csv')
    for path in csv_paths:
        data = pd.read_csv(path)

        data['utc'] = pd.to_datetime(data['utc'])
        data = data.sort_values('utc')

        threshold = 0.5
        data['pred'] = (data['predicted_proba'] > threshold).astype(int)
        data['group_id'] = ((data['utc'] - data['utc'].min()).dt.days // 5)
        data['target'] = (data['close'].shift(-1) >= data['close']).astype(int)

        name = path.split('/')[-1].split('_')[0]
        if name not in all_summary_dict:
            all_summary_dict[name] = []

        metrics_by_group = data.groupby('group_id').apply(compute_precision_recall).reset_index()

        type = path.split('/')[-1].split('_')[1].split('.')[0]
        type_bool = 1 if type == 'long' else 0

        fname = path.split('/')[-1].split('_')[0] + '_' + type + '_' + model_type + '_with_weight'

        row = {
            'file': fname,
            'accuracy_mean': metrics_by_group['accuracy'].mean(),
            'accuracy_median': metrics_by_group['accuracy'].median(),
            'pr_custom_mean': metrics_by_group[f'pr_{type}'].mean(),
            're_custom_mean': metrics_by_group[f're_{type}'].mean(),
            'f_beta_custom_mean': metrics_by_group[f'fbeta_{type}'].mean(),
            'pr_custom_median': metrics_by_group[f'pr_{type}'].median(),
            're_custom_median': metrics_by_group[f're_{type}'].median(),
            'f_beta_custom_median': metrics_by_group[f'fbeta_{type}'].median(),
        }
        all_summary_dict[name].append(row)

# Делаем итоговую таблицу (индексация по имени файла)

all_df_dict = dict()

for name in all_summary_dict.keys():
    all_df_dict[name] = pd.DataFrame(all_summary_dict[name]).set_index('file')

all_df_dict['Positive']


,accuracy_mean,accuracy_median,pr_mean,pr_median,re_mean,re_median,pr_custom_mean,re_custom_mean,f_beta_custom_mean,pr_custom_median,re_custom_median,f_beta_custom_median
file,,,,,,,,,,,,
Positive_long_catboost_no_weight,0.479450,0.480414,0.684245,0.681241,0.220411,0.213058,0.544630,0.255306,0.244092,0.548319,0.257926,0.238095
Positive_short_catboost_no_weight,0.605065,0.604692,0.540794,0.538175,0.253923,0.256568,0.263349,0.396339,0.263349,0.267352,0.393443,0.267352
Positive_long_lightgbm_no_weight,0.477876,0.481659,0.680421,0.677603,0.218865,0.216113,0.540822,0.253057,0.234655,0.542536,0.255185,0.233933
Positive_short_lightgbm_no_weight,0.603454,0.602911,0.537734,0.533762,0.241993,0.246824,0.254999,0.368639,0.254999,0.258929,0.362661,0.258929
Positive_long_xgboost_no_weight,0.477152,0.476410,0.688159,0.682074,0.210427,0.212702,0.546060,0.243259,0.243077,0.544256,0.251719,0.236919
Positive_short_xgboost_no_weight,0.601661,0.603526,0.537437,0.533865,0.218136,0.225000,0.264698,0.344499,0.264698,0.258303,0.343931,0.258303
Positive_long_catboost_with_weight,0.410332,0.413184,NaN,NaN,NaN,NaN,0.377557,0.001700,0.251986,0.250000,0.000564,0.000000
Positive_short_catboost_with_weight,0.590996,0.590827,NaN,NaN,NaN,NaN,0.383153,0.006216,0.383139,0.400000,0.003195,0.399988
Positive_short_lightgbm_with_weight,0.592431,0.592169,NaN,NaN,NaN,NaN,0.358006,0.029532,0.358003,0.369565,0.020408,0.369565


In [20]:
!pip install jinja2

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [24]:
print(all_df_dict['Positive'].to_latex())

\begin{tabular}{lrrrrrrrrrrrr}
\toprule
 & accuracy_mean & accuracy_median & pr_mean & pr_median & re_mean & re_median & pr_custom_mean & re_custom_mean & f_beta_custom_mean & pr_custom_median & re_custom_median & f_beta_custom_median \\
file &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
Positive_long_catboost_no_weight & 0.479450 & 0.480414 & 0.684245 & 0.681241 & 0.220411 & 0.213058 & 0.544630 & 0.255306 & 0.244092 & 0.548319 & 0.257926 & 0.238095 \\
Positive_short_catboost_no_weight & 0.605065 & 0.604692 & 0.540794 & 0.538175 & 0.253923 & 0.256568 & 0.263349 & 0.396339 & 0.263349 & 0.267352 & 0.393443 & 0.267352 \\
Positive_long_lightgbm_no_weight & 0.477876 & 0.481659 & 0.680421 & 0.677603 & 0.218865 & 0.216113 & 0.540822 & 0.253057 & 0.234655 & 0.542536 & 0.255185 & 0.233933 \\
Positive_short_lightgbm_no_weight & 0.603454 & 0.602911 & 0.537734 & 0.533762 & 0.241993 & 0.246824 & 0.254999 & 0.368639 & 0.254999 & 0.258929 & 0.362661 & 0.258929 \\
Positive_long_xgboost_no_weight & 0